# 20. The idea of SHAP — explained with the HAR model
**Part 4 — Explaining the forecasts (SHAP).** Before using SHAP on the LSTM, we see the idea on a model we already know: the HAR regression of notebook 09. For a linear model, SHAP can be computed by hand.

* **Needs:** `work/dataset.csv`. **Makes:** nothing new.

*Simple notebook series of the project 'This Time Is Different?'. Prepared with AI assistance (declared in `notes/ai_use.md`); a study notebook, not dissertation text.*

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
folder = '/content/drive/MyDrive/DataSets/ThisTimeIsDifferent/'

import warnings
warnings.filterwarnings('ignore')     # hide warning messages

In [2]:
from pandas import read_csv
dataset = read_csv(folder + 'work/dataset.csv', index_col=0, parse_dates=True)
dataset = dataset.loc[:'2023-08-31']        # main sample
print(dataset.shape)

(5243, 17)


## The HAR model of notebook 09 (trained on 2003–2006)

In [3]:
from numpy import log, exp
from statsmodels.api import OLS, add_constant
x = log(dataset[['rv_d', 'rv_w', 'rv_m']].clip(lower=0.00000001))
y = log(dataset['rv5_fwd'])
train_days = dataset.loc[:'2006-12-31'].dropna(subset=['rv5_fwd', 'rv_m']).index[:-5]
model = OLS(y.loc[train_days], add_constant(x.loc[train_days]))     # building
best_model = model.fit()                                             # training
best_model.params

const   -3.833383
rv_d     0.006900
rv_w     0.035312
rv_m     0.425291
dtype: float64

## The idea of SHAP
Think of a team that wins a bonus. **Shapley values** (from game theory) share the bonus fairly among the players, according to how much each one added. **SHAP** (Lundberg & Lee, 2017) does the same for a forecast:

* the 'bonus' = **this day's forecast − the average forecast**;
* the 'players' = the **inputs**;
* each input gets a **contribution**, and the contributions **add up** exactly to the bonus.

For a straight-line model the fair share has a simple formula:

**contribution of an input = its coefficient × (its value today − its average value)**

## One day: 16 August 2007 (inside the Global Financial Crisis)

In [4]:
x_day = x.loc['2007-08-16']
x_average = x.loc[train_days].mean()
coefficients = best_model.params[['rv_d', 'rv_w', 'rv_m']]
contributions = coefficients * (x_day - x_average)
contributions

rv_d    0.031910
rv_w    0.100466
rv_m    0.895291
dtype: float64

Check that they add up: the forecast for this day minus the average forecast must equal the sum of the contributions.

In [5]:
forecast_day = best_model.params['const'] + (coefficients * x_day).sum()
average_forecast = best_model.params['const'] + (coefficients * x_average).sum()
print('forecast - average forecast =', round(forecast_day - average_forecast, 4))
print('sum of the contributions    =', round(contributions.sum(), 4))

forecast - average forecast = 1.0277
sum of the contributions    = 1.0277


All three contributions are positive: today, last week and last month were all more volatile than usual, so each one **pushed the forecast up**. The **monthly** input pushed most.

## Shares
To compare inputs we use the size of each contribution (positive or negative does not matter) as a share of the total:

In [6]:
shares = contributions.abs() / contributions.abs().sum()
shares.round(3)

rv_d    0.031
rv_w    0.098
rv_m    0.871
dtype: float64

## A calm day for comparison: 15 May 2007

In [7]:
x_day = x.loc['2007-05-15']
contributions = coefficients * (x_day - x_average)
contributions

rv_d   -0.000434
rv_w    0.001758
rv_m    0.149909
dtype: float64

On this calmer day the contributions are much **smaller** (together about 0.15 instead of 1.03): the inputs were close to their usual level, so they moved the forecast only a little. On a day with lower-than-usual volatility they would be negative and push the forecast **down**.

## From HAR to the LSTM
* HAR uses only the **Irish** market — all its contributions belong to one 'channel' (domestic).
* The LSTM uses **three markets** and reads **22 days × 6 inputs = 132 inputs** for every forecast. It is not a straight line, so there is no simple formula; we use **GradientShap** (notebook 21), a SHAP method for neural networks.
* Adding the contributions by market gives the **channel shares**: which market the model was 'listening to'. This is the main measure of the explanation part of the study.